<a href="https://colab.research.google.com/github/lokeshraju1/harvard_artifacts/blob/main/Streamlit_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install streamlit streamlit-option-menu

In [8]:
%%writefile app.py

import sqlite3
import requests
import pandas as pd
import streamlit as st
from streamlit_option_menu import option_menu

conn = sqlite3.connect("Hardvard.db")
cursor = conn.cursor()

API_KEY = "c1a9f57b-2e39-442b-a6ca-b2559f0aa149"
object_url = "https://api.harvardartmuseums.org/object"

class_name = ("Photographs","Drawings","Prints","Paintings","Coins","Vessels","Archival Material")

#Create TABLE

def create_tables():
    cursor.execute("""CREATE TABLE IF NOT EXISTS artifact_metadata(
                    id INTEGER PRIMARY KEY ,
                    title TEXT,
                    culture TEXT,
                    period TEXT,
                    century TEXT,
                    medium TEXT,
                    dimensions TEXT,
                    description TEXT,
                    department TEXT,
                    classification TEXT,
                    accessionyear INTEGER,
                    accessionmethod TEXT
)""")
    cursor.execute("""CREATE TABLE IF NOT EXISTS artifact_media(
                    objectid INTEGER,
                    imagecount INTEGER,
                    mediacount INTEGER,
                    colorcount INTEGER,
                    rank INTEGER,
                    datebegin INTEGER,
                    dateend INTEGER
)""")
    cursor.execute("""CREATE TABLE IF NOT EXISTS artifact_colors(
                  objectid INTEGER,
                  color TEXT,
                  spectrum TEXT,
                  hue TEXT,
                  percent REAL,
                  css3 TEXT
)""")


create_tables()


#Data collection per classification


def classes(API_KEY,class_name):
    all_records = []


    for page in range(1, 26):
        params = {
            "apikey": API_KEY,
            "size": 100,
            "page": page,
            "classification": class_name
        }

        response = requests.get(object_url, params=params)

        object_data = response.json()
        records = object_data.get('records', [])
        all_records.extend(records)

    return all_records

# Collecting metadata, media and color details

def artifacts_details(records):

      metadata = []
      media = []
      colors = []

      for i in records:
        #artifact_metadata
          metadata.append(dict(
              id = i.get('id'),
              title = i.get('title'),
              culture =i.get('culture'),
              period =i.get('period'),
              century =i.get('century'),
              medium =i.get('medium'),
              dimensions = i.get('dimensions'),
              description =i.get('description'),
              department = i.get('department'),
              classification =i.get('classification'),
              accessionyear =i.get('accessionyear'),
              accessionmethod = i.get('accessionmethod')
          ))
        #artifact_media
          media.append(dict(
              objectid = i.get('objectid'),
              rank = i.get('rank'),
              imagecount =i.get('imagecount'),
              mediacount = i.get('mediacount'),
              colorcount = i.get('colorcount'),
              datebegin =i.get('datebegin'),
              dateend =i.get('dateend')
          ))
        #artifact_colors
          sub_list = i.get('colors', []) # Check if 'colors' key exists, otherwise use an empty list
          for j in sub_list:
              colors.append(dict(
                  objectid = i.get('objectid'),
                  color = j.get('color'),
                  spectrum = j.get('spectrum'),
                  hue = j.get('hue'), # Use .get() for safety
                  percent = j.get('percent'), # Use .get() for safety
                  css = j.get('css3') # Use .get() for safety
              ))

      return metadata,media,colors

#Data insert

def insert_values(meta,media,color):
      insert_meta = """INSERT INTO artifact_metadata values(?,?,?,?,?,?,?,?,?,?,?,?)"""
      insert_media = """INSERT INTO artifact_media values(?,?,?,?,?,?,?)"""
      insert_color  = """INSERT INTO artifact_colors values(?,?,?,?,?,?)"""

      for i in meta:
          values1 = (i['id'], i['title'], i['culture'], i['period'], i['century'], i['medium'], i['dimensions'], i['description'], i['department'], i['classification'], i['accessionyear'], i['accessionmethod'])
          cursor.execute(insert_meta,values1)

      for i in media:
          values2 = (i['objectid'], i['rank'], i['imagecount'], i['mediacount'], i['colorcount'], i['datebegin'], i['dateend'])
          cursor.execute(insert_media,values2)

      for i in color:
          values3 = (i['objectid'], i['color'], i['spectrum'], i['hue'], i['percent'], i['css'])
          cursor.execute(insert_color,values3)

      conn.commit()

#Streamlit UI

st.set_page_config(layout="wide")



st.markdown("<h1 style='text-align: center; color: rainbow;'>🏛️Artifacts Collection</h1>", unsafe_allow_html=True)


classification = st.selectbox("Enter a classification name :",("Photographs","Drawings","Prints","Paintings","Coins","Vessels","Archival Material","Sculpture","Textile Arts","Fragments","Manuscripts","Seals","Straus Materials"),)
button = st.button("Collect data")
menu = option_menu(None,["Select Your Choice","Migrate to SQL","SQL Queries"], orientation="horizontal")




if button:
    if classification != '':
        records = classes(API_KEY,classification)
        meta ,media,color = artifacts_details(records)
        c1,c2,c3 = st.columns(3)
        with c1:
              st.header("Metadata")
              st.json(meta)
        with c2:
              st.header("Media")
              st.json(media)
        with c3:
              st.header("Colours")
              st.json(color)

    else:
          st.error("Kindly enter a classification")



if menu == 'Migrate to SQL':

        cursor.execute("select distinct(classification) from artifact_metadata")
        result = cursor.fetchall()
        classes_list = [i[0] for i in result]


        st.subheader("Insert the collected data")
        if st.button("Insert"):
          if classification not in classes_list:

                records = classes(API_KEY,classification)
                meta,media,color = artifacts_details(records)
                insert_values(meta,media,color)
                st.success("Data Inserted successfully")

                st.header("Inserted Data:")
                st.divider()

                st.subheader("Artifacts Metadata")
                cursor.execute("select * from artifact_metadata")
                result1 = cursor.fetchall()
                columns = [i[0] for i in cursor.description]
                df1 = pd.DataFrame(result1,columns=columns)
                st.dataframe(df1)

                st.subheader("Artifacts Media")
                cursor.execute("select * from artifact_media")
                result2 = cursor.fetchall()
                columns = [i[0] for i in cursor.description]
                df2 = pd.DataFrame(result2,columns=columns)
                st.dataframe(df2)

                st.subheader("Artifacts Colors")
                cursor.execute("select * from artifact_colors")
                result3 = cursor.fetchall()
                columns = [i[0] for i in cursor.description]
                df3 = pd.DataFrame(result3,columns=columns)
                st.dataframe(df3)
          else:
            st.error("Classification already exists!! Kindly try a different class ! ")




elif menu == "SQL Queries":

        option = st.selectbox("Queries",
                ("1.List all artifacts from the 11th century belonging to Byzantine culture.",
                  "2.What are the unique cultures represented in the artifacts?",
                  "3.List all artifacts from the Archaic Period",
                  "4.List artifact titles ordered by accession year in descending order.",
                  "5.How many artifacts are there per department?",
                  "6.Which artifacts have more than 3 images?",
                  "7.What is the average rank of all artifacts?",
                  "8.Which artifacts have a higher mediacount than colorcount?",
                  "9.List all artifacts created between 1500 and 1600",
                  "10.How many artifacts have no media files?",
                  "11.What are all the distinct hues used in the dataset?",
                  "12.What are the top 5 most used colors by frequency?",
                  "13.What is the average coverage percentage for each hue?",
                  "14.List all colors used for a given artifact ID.",
                  "15.What is the total number of color entries in the dataset?",
                  "16.List artifact titles and hues for all artifacts belonging to the Byzantine culture.",
                  "17.List each artifact title with its associated hues.",
                  "18.Get artifact titles, cultures, and media ranks where the period is not null.",
                  "19.Find artifact titles ranked in the top 10 that include the color hue Grey",
                  "20.How many artifacts exist per classification, and what is the average media count for each?",
                  "21.Which cultures have artifacts spanning more than 3 centuries?",
                  "22.What are the top 3 hues used in artifacts from the Greek culture?",
                  "23.Which artifacts have both more than 2 images and a rank above 80?",
                  "24.Which departments have artifacts with the hue Blue?",
                  "25.What is the average number of colors per artifact for each classification?",),index =None,placeholder="Select a query")


        if option == "1.List all artifacts from the 11th century belonging to Byzantine culture.":
            cursor.execute("""select * from artifact_metadata where century = '11th century' and culture = 'Byzantine' """)
            result = cursor.fetchall()
            df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
            st.dataframe(df)

        elif option == "2.What are the unique cultures represented in the artifacts?":
           cursor.execute("""select distinct(culture) from artifact_metadata""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "3.List all artifacts from the Archaic Period":
           cursor.execute("""SELECT * from artifact_metadata where period='Archaic_Period'""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "4.List artifact titles ordered by accession year in descending order.":
           cursor.execute("SELECT * from artifact_metadata order by accessionyear desc")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "5.How many artifacts are there per department?":
           cursor.execute("""SELECT department, count(*) from artifact_metadata GROUP BY department""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "6.Which artifacts have more than 3 images?":
           cursor.execute("""SELECT * from artifact_media where imagecount > 3""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "7.What is the average rank of all artifacts?":
           cursor.execute("""SELECT AVG(rank) FROM artifact_media""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "8.Which artifacts have a higher mediacount than colorcount?":
           cursor.execute("""SELECT * FROM artifact_media where mediacount > colorcount """)
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "9.List all artifacts created between 1500 and 1600":
           cursor.execute("""SELECT * FROM artifact_media where datebegin >= 1500 AND dateend <= 1600 and dateend !=0""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "10.How many artifacts have no media files?":
           cursor.execute("""SELECT COUNT(*) FROM artifact_media WHERE mediacount = 0""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "11.What are all the distinct hues used in the dataset?":
           cursor.execute("""SELECT DISTINCT hue FROM artifact_colors""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "12.What are the top 5 most used colors by frequency?":
           cursor.execute("""SELECT color, COUNT(*) AS frequency FROM artifact_colors GROUP BY color ORDER BY frequency DESC LIMIT 5""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "13.What is the average coverage percentage for each hue?":
           cursor.execute("""SELECT hue, AVG(percent) FROM artifact_colors GROUP BY hue""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "14.List all colors used for a given artifact ID.":
           # Replace '[artifact_id]' with a placeholder '?' and provide a sample ID
           artifact_id_to_query = 379076  # Replace with an actual artifact ID
           cursor.execute("SELECT color, spectrum, hue, percent, css3 FROM artifact_colors WHERE objectid = ?", (artifact_id_to_query,),)
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "15.What is the total number of color entries in the dataset?":
           cursor.execute("""SELECT COUNT(*) FROM artifact_colors""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "16.List artifact titles and hues for all artifacts belonging to the Byzantine culture.":
           cursor.execute("""SELECT am.title, ac.hue FROM artifact_metadata am JOIN artifact_colors ac ON am.id = ac.objectid WHERE am.culture = 'Byzantine'""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "17.List each artifact title with its associated hues.":
           cursor.execute("""SELECT am.title, ac.hue FROM artifact_metadata am JOIN artifact_colors ac ON am.id = ac.objectid""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "18.Get artifact titles, cultures, and media ranks where the period is not null.":
           cursor.execute("""SELECT am.title, am.culture, ami.rank FROM artifact_metadata am JOIN artifact_media ami ON am.id = ami.objectid WHERE am.period IS NOT NULL""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "19.Find artifact titles ranked in the top 10 that include the color hue Grey":
           cursor.execute("""SELECT DISTINCT am.title FROM artifact_metadata am JOIN artifact_media ami ON am.id = ami.objectid JOIN artifact_colors ac ON am.id = ac.objectid WHERE ami.rank <= 10 AND ac.hue = 'Grey'""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "20.How many artifacts exist per classification, and what is the average media count for each?":
           cursor.execute("""SELECT am.classification, COUNT(am.id) AS artifact_count, AVG(ami.mediacount) AS average_media_count FROM artifact_metadata am JOIN artifact_media ami ON am.id = ami.objectid GROUP BY am.classification""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "21.Which cultures have artifacts spanning more than 3 centuries?":
           cursor.execute("""SELECT culture, COUNT(DISTINCT century) AS century_span FROM artifact_metadata WHERE century IS NOT NULL GROUP BY culture HAVING century_span > 3 ORDER BY century_span DESC""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "22.What are the top 3 hues used in artifacts from the Greek culture?":
           cursor.execute("""SELECT c.hue, COUNT(*) AS usage_count FROM artifact_metadata m JOIN artifact_colors c ON m.id = c.objectid WHERE m.culture = 'Greek' GROUP BY c.hue ORDER BY usage_count DESC LIMIT 3""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "23.Which artifacts have both more than 2 images and a rank above 80?":
           cursor.execute("""SELECT m.title, me.imagecount, me.rank FROM artifact_metadata m JOIN artifact_media me ON m.id = me.objectid WHERE me.imagecount > 2 AND me.rank > 80""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "24.Which departments have artifacts with the hue Blue?":
           cursor.execute("""SELECT DISTINCT m.department FROM artifact_metadata m JOIN artifact_colors c ON m.id = c.objectid WHERE c.hue = 'Blue'""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

        elif option == "25.What is the average number of colors per artifact for each classification?":
           cursor.execute("""SELECT m.classification, AVG(color_count.total_colors) AS avg_colors FROM artifact_metadata m JOIN (SELECT objectid, COUNT(*) AS total_colors FROM artifact_colors GROUP BY objectid ) AS color_count ON m.id = color_count.objectid GROUP BY m.classification ORDER BY avg_colors DESC""")
           result = cursor.fetchall()
           df = pd.DataFrame(result,columns = [i[0] for i in cursor.description])
           st.dataframe(df)

Overwriting app.py


In [9]:
# @title Setup code
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

--2025-12-24 08:52:55--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2025.11.1/cloudflared-linux-amd64 [following]
--2025-12-24 08:52:55--  https://github.com/cloudflare/cloudflared/releases/download/2025.11.1/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/955e9d1b-ac5e-4188-8867-e5f53958a8fe?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-12-24T09%3A52%3A59Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-12-24

In [10]:
!streamlit run /content/app.py &>/content/logs.txt &

In [11]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://partition-browser-spencer-true.trycloudflare.com


In [12]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("Hardvard.db")
cursor = conn.cursor()

# Re-create tables if they don't exist to ensure the schema is present
def create_tables():
    cursor.execute("""CREATE TABLE IF NOT EXISTS artifact_metadata(
                    id INTEGER PRIMARY KEY ,
                    title TEXT,
                    culture TEXT,
                    period TEXT,
                    century TEXT,
                    medium TEXT,
                    dimensions TEXT,
                    description TEXT,
                    department TEXT,
                    classification TEXT,
                    accessionyear INTEGER,
                    accessionmethod TEXT
)""")
    cursor.execute("""CREATE TABLE IF NOT EXISTS artifact_media(
                    objectid INTEGER,
                    imagecount INTEGER,
                    mediacount INTEGER,
                    colorcount INTEGER,
                    rank INTEGER,
                    datebegin INTEGER,
                    dateend INTEGER
)""")
    cursor.execute("""CREATE TABLE IF NOT EXISTS artifact_colors(
                  objectid INTEGER,
                  color TEXT,
                  spectrum TEXT,
                  hue TEXT,
                  percent REAL,
                  css3 TEXT
)""")
    conn.commit()

create_tables()

# Now execute the query after ensuring tables exist
cursor.execute("SELECT m.title, me.imagecount, me.rank FROM artifact_metadata m JOIN artifact_media me ON m.id = me.objectid WHERE me.imagecount > 2 AND me.rank > 80")

result = cursor.fetchall()
df = pd.DataFrame(result, columns=[i[0] for i in cursor.description])
display(df)

conn.close()

,title,imagecount,rank
